# Can I trust this call? QC & read evidence

**Utility:** a virus lighting up in the summary is not automatically real. ViralScan
ships diagnostics to separate genuine infection from artifacts: sibling
cross-mapping flags, accession breadth, host-virus ambiguity fraction, and read-level
tracing via `viralscan evidence`.

## Sibling cross-mapping (EM bleed between near-identical viruses)

HHV-6A/6B (~95% identity) and HSV-1/2 share sequence. When one sibling dominates, the
global EM allocates a little of its mass to the other, producing a residual that
looks like a weak co-infection. `check_sibling_crossmapping` flags the weaker member
when the UMI ratio exceeds the threshold (default 50:1).

In [ ]:
from viralscan.scripts.detection import check_sibling_crossmapping

# Dominant HHV-6B with a weak HHV-6 residual (211:1) -> flagged.
virus_stats = {
    'Human herpesvirus 6b': {'total_umi': 6944},
    'Human herpesvirus 6':  {'total_umi': 33},
}
notes = check_sibling_crossmapping(virus_stats)
for virus, note in notes.items():
    print(f'{virus}: {note}')

The weaker sibling is annotated `possible_em_bleed` in `viral_summary.tsv`
(`sibling_crossmap_note`) so you do not over-interpret it as genuine co-infection.

## Accession breadth and host-virus ambiguity

`compute_stats` reports two per-virus trust signals:

- **`accession_breadth`** — fraction of the virus's reference accessions with ≥1 UMI.
  Genuine infection spreads across the genome; **EVE artifacts concentrate on 1–2
  host-integrated loci** (breadth near 0).
- **`host_viral_ambig_fraction`** — fraction of viral UMI that mapped ambiguously to
  both host and virus. A high value signals reads originating from host genomic
  regions (e.g. endogenous viral elements in expressed host genes).

In [ ]:
import numpy as np, anndata as ad, pandas as pd, scipy.sparse as sp
from viralscan.scripts.detection import compute_stats

# Virus with 3 accessions but signal on only 1 (low breadth = EVE-like).
X = np.array([[8, 0, 0, 50], [4, 0, 0, 40], [0, 0, 0, 30]], dtype=float)
adata = ad.AnnData(sp.csr_matrix(X))
adata.obs_names = ['bc0', 'bc1', 'bc2']
adata.var_names = ['acc1', 'acc2', 'acc3', 'HOST']

stats, _ = compute_stats(adata, {'acc1': 1, 'acc2': 1, 'acc3': 1},
                         {'AnellovirusX': ['acc1', 'acc2', 'acc3']}, [])
s = stats['AnellovirusX']
print('accession_breadth        :', s['accession_breadth'], '(1 of 3 accessions)')
print('host_viral_ambig_fraction:', s['host_viral_ambig_fraction'])

Low breadth on a single accession is a red flag to inspect the reads before claiming
infection.

## Trace the reads behind a call: `viralscan evidence`

For a completed run, `evidence` extracts the (barcode, UMI) reads assigned to viral
genes, BLASTs them, and scores them — the audit trail for a called virus.

```bash
viralscan evidence --run-dir out/sample/ --output out/sample/evidence/ \
                   --virus EBV --blast
# -> per-read BLAST identity, so you can confirm reads are viral, not host look-alikes.
```

> **Host-homology caveat (from the paper):** ViralScan uses a cDNA-only host
> reference, which covers spliced exons but not intronic/intergenic sequence. Reads
> from GRCh38 non-coding regions that resemble a viral reference cannot be attributed
> to the host and can appear as spurious viral signal. For whole-blood, high-intron,
> or nuclear-RNA libraries, run an orthogonal alignment to a combined genome as a
> specificity control.

## Summary — a trust checklist for any viral call

1. Is it a `possible_em_bleed` sibling of a dominant virus? (cross-mapping note)
2. Is `accession_breadth` high, or concentrated on 1–2 loci (EVE-like)?
3. Is `host_viral_ambig_fraction` high (host-homology artifact)?
4. Do the reads BLAST as viral (`viralscan evidence`)?